In [1]:
# ===== RÉGLAGES DE L'ESSAI : on ne change qu'UNE chose à la fois =====
ESSAI = "epochs6"        # nom de l'essai (utilisé dans les noms de fichiers)
NB_EPOCHS = 6            # v2_05 : 4
LEARNING_RATE = 2e-5     # v2_05 : 2e-5
MAX_LENGTH = 384         # v2_05 : 384
TAILLE_LOT = 16          # v2_05 : 16 (par GPU)

In [2]:
import glob, os, sys, math, json
import numpy as np
import pandas as pd
import torch
import transformers

DOSSIER = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
sys.path.append(os.path.dirname(glob.glob("/kaggle/input/**/metrics.py", recursive=True)[0]))
from metrics import GROUPES, evaluer, rapport_par_groupe, probas_vers_predictions, chercher_seuils

train = pd.read_csv(f"{DOSSIER}/train.csv", dtype={"id": str})
val = pd.read_csv(f"{DOSSIER}/val.csv", dtype={"id": str})
test = pd.read_csv(f"{DOSSIER}/test.csv", dtype={"id": str})

print("Essai :", ESSAI)
print("Train :", train.shape, "| Validation :", val.shape, "| Test :", test.shape)
print("GPU :", torch.cuda.device_count(), "x",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN")

Essai : epochs6
Train : (13993, 11) | Validation : (3006, 11) | Test : (2983, 11)
GPU : 2 x Tesla T4


In [3]:
from datasets import Dataset, Sequence, Value
from transformers import AutoTokenizer

NOM_MODELE = "allenai/scibert_scivocab_uncased"
tokenizer = AutoTokenizer.from_pretrained(NOM_MODELE)

def vers_dataset(df, avec_labels=True):
    donnees = {"text": df["text"].tolist()}
    if avec_labels:
        donnees["labels"] = df[GROUPES].values.astype("float32").tolist()
    ds = Dataset.from_dict(donnees)
    if avec_labels:
        ds = ds.cast_column("labels", Sequence(Value("float32")))
    return ds.map(lambda lot: tokenizer(lot["text"], truncation=True, max_length=MAX_LENGTH),
                  batched=True, remove_columns=["text"])

ds_train = vers_dataset(train)
ds_val = vers_dataset(val)
ds_test = vers_dataset(test, avec_labels=False)   # sans labels -> aucun score test possible
print(ds_train)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/228k [00:00<?, ?B/s]

Casting the dataset:   0%|          | 0/13993 [00:00<?, ? examples/s]

Map:   0%|          | 0/13993 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/3006 [00:00<?, ? examples/s]

Map:   0%|          | 0/3006 [00:00<?, ? examples/s]

Map:   0%|          | 0/2983 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 13993
})


In [4]:
from transformers import AutoModelForSequenceClassification

modele = AutoModelForSequenceClassification.from_pretrained(
    NOM_MODELE, num_labels=len(GROUPES), problem_type="multi_label_classification",
    id2label=dict(enumerate(GROUPES)), label2id={g: i for i, g in enumerate(GROUPES)})

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  442MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those

In [5]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

def calculer_metriques(eval_pred):
    logits, labels = eval_pred
    probas = 1 / (1 + np.exp(-logits))
    return evaluer(labels.astype(int), probas_vers_predictions(probas))

nb_gpus = max(1, torch.cuda.device_count())
pas_par_epoch = math.ceil(len(ds_train) / (TAILLE_LOT * nb_gpus))
pas_warmup = int(0.1 * pas_par_epoch * NB_EPOCHS)
print("Pas par epoch :", pas_par_epoch, "| pas de warmup :", pas_warmup)

args = TrainingArguments(
    output_dir="/kaggle/working/scibert", num_train_epochs=NB_EPOCHS,
    learning_rate=LEARNING_RATE, per_device_train_batch_size=TAILLE_LOT, per_device_eval_batch_size=64,
    warmup_steps=pas_warmup, weight_decay=0.01, fp16=True,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="micro_f1", save_total_limit=1,
    logging_strategy="epoch", report_to="none", seed=42)

trainer = Trainer(model=modele, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=calculer_metriques)
trainer.train()

Pas par epoch : 438 | pas de warmup : 262


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Micro F1,Macro F1,Hamming Loss,Subset Accuracy,Runtime,Samples Per Second,Steps Per Second
1,0.436599,0.210088,0.882394,0.678592,0.034639,0.794744,36.767500,81.757000,0.653000
2,0.177611,0.178194,0.891234,0.745391,0.032144,0.802728,36.678200,81.956000,0.654000
3,0.131937,0.171877,0.894438,0.743383,0.031645,0.804059,36.552200,82.239000,0.657000
4,0.099732,0.177599,0.894971,0.762167,0.031437,0.807718,36.806200,81.671000,0.652000
5,0.074432,0.191386,0.892482,0.762336,0.032352,0.801065,36.772800,81.745000,0.653000
6,0.060460,0.196323,0.890330,0.760174,0.033059,0.798403,36.606000,82.118000,0.656000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2628, training_loss=0.1634617301789956, metrics={'train_runtime': 3439.1844, 'train_samples_per_second': 24.412, 'train_steps_per_second': 0.764, 'total_flos': 1.5970054846426944e+16, 'train_loss': 0.1634617301789956, 'epoch': 6.0})

In [6]:
pred_val = trainer.predict(ds_val)
probas_val = 1 / (1 + np.exp(-pred_val.predictions))
y_val = pred_val.label_ids.astype(int)

scores_05 = evaluer(y_val, probas_vers_predictions(probas_val))
seuils = chercher_seuils(y_val, probas_val)
scores_seuils = evaluer(y_val, probas_vers_predictions(probas_val, seuils))

meilleure_epoch = round(int(trainer.state.best_model_checkpoint.split("-")[-1]) / pas_par_epoch)
print("Essai :", ESSAI, "| meilleure epoch :", meilleure_epoch)
print("Seuils choisis :", dict(zip(GROUPES, seuils.tolist())))
print(f"{'':16s} {'seuil 0.5':>10s} {'seuils/groupe':>14s}")
for nom in scores_05:
    print(f"{nom:16s} {scores_05[nom]:>10.4f} {scores_seuils[nom]:>14.4f}")
print()
print(rapport_par_groupe(y_val, probas_vers_predictions(probas_val, seuils)))

reglages = {"NB_EPOCHS": NB_EPOCHS, "LEARNING_RATE": LEARNING_RATE,
            "MAX_LENGTH": MAX_LENGTH, "TAILLE_LOT": TAILLE_LOT}
with open(f"/kaggle/working/scibert_{ESSAI}_val.json", "w") as f:
    json.dump({"reglages": reglages, "meilleure_epoch": meilleure_epoch, "seuil_05": scores_05,
               "seuils": seuils.tolist(), "seuils_par_groupe": scores_seuils,
               "historique": trainer.state.log_history}, f, indent=2)
np.save(f"/kaggle/working/scibert_{ESSAI}_val_probas.npy", probas_val)
print(f"Sauvegardé : scibert_{ESSAI}_val.json, scibert_{ESSAI}_val_probas.npy")

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Essai : epochs6 | meilleure epoch : 4
Seuils choisis : {'cs': 0.55, 'math': 0.4, 'physics': 0.65, 'q-bio': 0.25, 'q-fin': 0.45, 'stat': 0.2, 'eess': 0.35, 'econ': 0.6}
                  seuil 0.5  seuils/groupe
micro_f1             0.8950         0.8946
macro_f1             0.7622         0.7745
hamming_loss         0.0314         0.0319
subset_accuracy      0.8077         0.8051

              precision    recall  f1-score   support

          cs       0.91      0.93      0.92       973
        math       0.87      0.87      0.87       717
     physics       0.97      0.95      0.96      1497
       q-bio       0.66      0.72      0.69        58
       q-fin       0.80      0.64      0.71        55
        stat       0.63      0.70      0.66       146
        eess       0.54      0.75      0.63       125
        econ       0.84      0.69      0.76        52

   micro avg       0.89      0.90      0.89      3623
   macro avg       0.78      0.78      0.77      3623
weighted avg       0

In [7]:
import shutil

pred_test = trainer.predict(ds_test)
probas_test = 1 / (1 + np.exp(-pred_test.predictions))
np.save(f"/kaggle/working/scibert_{ESSAI}_test_probas.npy", probas_test)
test[["id"]].to_csv(f"/kaggle/working/scibert_{ESSAI}_test_ids.csv", index=False)
print("Probabilités test sauvegardées :", probas_test.shape)

shutil.rmtree("/kaggle/working/scibert", ignore_errors=True)   # supprime les checkpoints

Probabilités test sauvegardées : (2983, 8)
